## Problem: Write a GAN

### Problem Statement
Implement a **Generative Adversarial Network (GAN)** by completing the required sections. The GAN consists of a **Generator** that creates fake data and a **Discriminator** that classifies data as real or fake.

### Requirements
1. **Define the Generator Class**:
   - **Purpose**: Generate fake data that mimics the real data distribution.
   - **Layers**:
     - Start with a fully connected layer to map the latent space (random noise) to a higher-dimensional space.
     - Use activation functions like `ReLU` to introduce non-linearity.
     - Add additional layers to process the data and refine its structure.
     - The final layer should output data in the target shape. Use an activation function like `Tanh` for scaling.
   - **Forward Pass**: Implement the forward method to pass the input through the defined layers.

2. **Define the Discriminator Class**:
   - **Purpose**: Classify data as real or fake.
   - **Layers**:
     - Use fully connected layers to process the input and extract features.
     - Apply activation functions like `LeakyReLU` to prevent dead neurons and stabilize training.
     - The final layer should output a single probability (real or fake) using a `Sigmoid` activation.
   - **Forward Pass**: Implement the forward method to process the input through the layers.

3. **Train the GAN**:
   - Alternate between training the Generator and Discriminator.
   - Use binary cross-entropy loss for both models.
   - Monitor the loss and generated samples during training.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [29]:
# Define the Generator
#TODO: Implement the Generator class
class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        # Add layers for the generator
        self.module = nn.Sequential(
            nn.Linear(input_dim, 256), 
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh(),
        )
    
    def forward(self, x):
        # Define the forward pass for the generator
        return self.module(x)

# Define the Discriminator
#TODO: Implement the Discriminator class
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        # Add layers for the discriminator
        self.module = nn.Sequential(
            nn.Linear(input_dim, 256), 
            nn.LeakyReLU(0.2),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        # Define the forward pass for the discriminator
        return self.module(x)

In [30]:
# Generate synthetic data for training
torch.manual_seed(27)
real_data = torch.rand(1000, 1) * 2 - 1  # 100 samples in the range [-1, 1]

# Initialize models, loss, and optimizers
latent_dim = 10
data_dim = 1
G = Generator(latent_dim, data_dim)
D = Discriminator(data_dim)

criterion = nn.CrossEntropyLoss()
optimizer_G = optim.Adam(G.parameters(), lr=0.001)
optimizer_D = optim.Adam(D.parameters(), lr=0.001)

In [32]:
# Training loop
epochs = 1000
for epoch in range(epochs):
    # Train Discriminator
    latent_samples = torch.randn(real_data.size(0), latent_dim)
    fake_data = G(latent_samples).detach()
    real_labels = torch.ones(real_data.size(0), dtype=torch.long)
    fake_labels = torch.zeros(real_data.size(0), dtype=torch.long)

    optimizer_D.zero_grad()
    real_loss = criterion(D(real_data), real_labels)
    fake_loss = criterion(D(fake_data), fake_labels)
    loss_D = real_loss + fake_loss
    loss_D.backward()
    optimizer_D.step()

    # Train Generator
    latent_samples = torch.randn(real_data.size(0), latent_dim)
    fake_data = G(latent_samples)
    optimizer_G.zero_grad()
    loss_G = criterion(D(fake_data), real_labels)
    loss_G.backward()
    optimizer_G.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 1 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] - Loss D: {loss_D.item():.4f}, Loss G: {loss_G.item():.4f}")

Epoch [1/1000] - Loss D: 1.3045, Loss G: 0.7869
Epoch [2/1000] - Loss D: 1.2897, Loss G: 0.8015
Epoch [3/1000] - Loss D: 1.2984, Loss G: 0.8020
Epoch [4/1000] - Loss D: 1.3045, Loss G: 0.7893
Epoch [5/1000] - Loss D: 1.3016, Loss G: 0.8059
Epoch [6/1000] - Loss D: 1.3133, Loss G: 0.7955
Epoch [7/1000] - Loss D: 1.3219, Loss G: 0.7955
Epoch [8/1000] - Loss D: 1.3113, Loss G: 0.7812
Epoch [9/1000] - Loss D: 1.3240, Loss G: 0.7764
Epoch [10/1000] - Loss D: 1.3507, Loss G: 0.7918
Epoch [11/1000] - Loss D: 1.3651, Loss G: 0.8101
Epoch [12/1000] - Loss D: 1.3719, Loss G: 0.8314
Epoch [13/1000] - Loss D: 1.3568, Loss G: 0.7825
Epoch [14/1000] - Loss D: 1.3823, Loss G: 0.7467
Epoch [15/1000] - Loss D: 1.3714, Loss G: 0.7573
Epoch [16/1000] - Loss D: 1.4058, Loss G: 0.7926
Epoch [17/1000] - Loss D: 1.3957, Loss G: 0.7845
Epoch [18/1000] - Loss D: 1.4146, Loss G: 0.7603
Epoch [19/1000] - Loss D: 1.4185, Loss G: 0.7421
Epoch [20/1000] - Loss D: 1.4255, Loss G: 0.7023
Epoch [21/1000] - Loss D: 1.4

In [13]:
# Generate new samples with the trained Generator
latent_samples = torch.randn(5, latent_dim)
with torch.no_grad():
    generated_data = G(latent_samples)
    print(f"Generated data: {generated_data.tolist()}")

Generated data: [[-0.6785176396369934], [0.8586157560348511], [0.39853808283805847], [0.8134640455245972], [-0.9402514696121216]]
